# LightGBM + exact-time GraphSAGE: validation-driven conditional collision routing

TRAIN-only exact-vector registry. Validation-only threshold selection with fixed FP/FN budgets; TEST is opened after policy freeze. Frozen LightGBM and preprocessing; newly trained exact-microsecond GraphSAGE and full exact-time graph.

Do not attach the old v4 GraphSAGE score cache: this notebook uses a different checkpoint. Enable Kaggle GPUs for batched GraphSAGE inference across all visible GPUs; CPU worker processes build ego graphs concurrently from independent read-only SQLite connections. Host shard decoding and SQLite lookups remain CPU work. If the run is interrupted, atomic per-TEST-shard checkpoints are saved under `/kaggle/working/cascade_run_exact_time_lgbm_conditional_collision_resumable_v2/test_shards/`. Validation and TEST checkpoints are written after every shard. A rerun in the same working session validates and skips completed shards. To resume in a fresh session, attach the prior notebook output as an optional third Kaggle input and set `RESUME_FROM` in the setup cell to that output folder; checkpoints are valid across CPU/GPU configurations. The standalone sample selected threshold was about 0.9002, while the cascade middle band intentionally keeps 0.95; only the full validation cascade can show whether the collision policy helps.


In [ ]:
from pathlib import Path
import json, os, sys, tarfile, shutil

MODEL_NAME = 'lgbm'
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
OUTPUT = WORK / 'cascade_run_exact_time_lgbm_conditional_collision_resumable_v2'
# Optional: previous run output folder with TRAIN registry and/or validation_shards/test_shards; policy is not required after a validation timeout.
RESUME_FROM = Path(
    "/kaggle/input/datasets/ngkimngn/lgbm-two-stage-v9/"
    "cascade_run_exact_time_lgbm_conditional_collision_resumable_v2"
)
WORK.mkdir(parents=True, exist_ok=True)

# Kaggle may expose the dataset as extracted files or as a single archive.
bundle = INPUT / 'datasets/ngkimngn/optc-cascade-exact-time-graphsage'
required = ('bundle_manifest.json', 'provenance/graph.sqlite', 'provenance/model_numpy.npz',
            'code/cascade_runtime.py', f'models/{MODEL_NAME}_metrics.json')
def is_exact_time_bundle(path):
    if not all((path / name).is_file() for name in required):
        return False
    metadata = json.loads((path / 'bundle_manifest.json').read_text())
    return (metadata.get('schema') == 5 and metadata.get('experiment') == 'exact_time_graphsage_full_cascade'
            and metadata.get('context_policy', {}).get('timestamp_quantum_us') == 1
            and metadata.get('context_policy', {}).get('checkpoint_training_quantum_us') == 1
            and metadata.get('context_policy', {}).get('checkpoint_retrained') is True)

if not is_exact_time_bundle(bundle):
    extracted = [p.parent for p in INPUT.glob('**/bundle_manifest.json')
                 if is_exact_time_bundle(p.parent)]
    if len(extracted) == 1:
        bundle = extracted[0]
    else:
        archives = sorted(INPUT.glob('**/provenance-cascade-exact-time-graphsage.tar.gz'))
        if len(archives) != 1:
            raise RuntimeError(f'Expected one extracted exact-time bundle or archive; found {len(extracted)} extracted, {len(archives)} archives')
        bundle = WORK / 'cascade_bundle_exact_time_graphsage'
        bundle.mkdir(exist_ok=True)
        with tarfile.open(archives[0], 'r:gz') as tar:
            for member in tar.getmembers():
                target = (bundle / member.name).resolve()
                if not target.is_relative_to(bundle.resolve()) or not member.isfile():
                    raise ValueError(f'Unsafe bundle member: {member.name}')
            tar.extractall(bundle)
if not is_exact_time_bundle(bundle):
    raise RuntimeError('This notebook requires the full exact-time GraphSAGE bundle (schema 5)')
bundle_meta = json.loads((bundle / 'bundle_manifest.json').read_text())
assert bundle_meta['context_policy']['timestamp_quantum_us'] == 1
assert bundle_meta['context_policy']['checkpoint_retrained'] is True
assert bundle_meta['context_policy']['checkpoint_training_quantum_us'] == 1
cache_meta = json.loads((bundle / 'provenance/ego_cache_manifest.json').read_text())
training_config = json.loads((bundle / 'provenance/model_training_config.json').read_text())
if cache_meta['manifest_sha256'] != bundle_meta['context_policy']['ego_cache_manifest_sha256']:
    raise ValueError('GraphSAGE training-cache identity mismatch')
if training_config['cache_manifest_sha256'] != cache_meta['manifest_sha256']:
    raise ValueError('GraphSAGE checkpoint/cache mismatch')
if training_config['time_policy']['quantum_us'] != 1 or cache_meta['time_policy']['quantum_us'] != 1:
    raise ValueError('GraphSAGE checkpoint is not exact-time trained')
if training_config['relations'] != cache_meta['relations'] or training_config['attributes'] != len(cache_meta['attributes']):
    raise ValueError('GraphSAGE checkpoint feature/relation schema mismatch')
if cache_meta['relations'] != ['creates', 'communicates', 'communicates_reverse', 'hosts'] or len(cache_meta['attributes']) != 40:
    raise ValueError('Unexpected GraphSAGE node/edge schema')
sys.path.insert(0, str(bundle / 'code'))

host_root = INPUT / 'datasets/nhatnguyen2101/optc-host-ml-full-input'
manifest = host_root / 'manifest.json'
if not (manifest.is_file() and (host_root / 'assets/vocab.json').is_file()):
    candidates = [p for p in INPUT.glob('**/manifest.json')
                  if (p.parent / 'assets/vocab.json').is_file()]
    if len(candidates) == 1:
        manifest = candidates[0]
    else:
        archives = sorted(INPUT.glob('**/host-input.tar'))
        if len(archives) != 1:
            raise RuntimeError(f'Expected one extracted Host dataset or archive; found {len(candidates)} extracted, {len(archives)} archives')
        host_root = WORK / 'host_shards'
        host_root.mkdir(exist_ok=True)
        with tarfile.open(archives[0], 'r') as tar:
            for member in tar.getmembers():
                target = (host_root / member.name).resolve()
                if not target.is_relative_to(host_root.resolve()) or not member.isfile():
                    raise ValueError(f'Unsafe Host archive member: {member.name}')
            tar.extractall(host_root)
        manifest = host_root / 'manifest.json'

print('Host model:', MODEL_NAME)
print('Cascade bundle:', bundle)
print('Host shards:', manifest)
print('Output:', OUTPUT)

# Old v4 score caches are checkpoint-incompatible; score with this bundle and resume per shard.
score_cache = None
print('GraphSAGE score source: exact-time checkpoint in bundle; per-shard cache in OUTPUT')


In [ ]:
%%writefile /kaggle/working/collision_aware_runtime.py
"""Exact-time retrained GraphSAGE collision-aware routing experiment.

The exact binary CSR row is the key. Labels are consulted only while building
the full-TRAIN and fit-sample diagnostic registries, never in routing.
"""
from __future__ import annotations

import argparse
from collections import Counter, defaultdict
import json
import os
from pathlib import Path
import sqlite3

import numpy as np

from cascade_runtime import counts, metrics, sha, safe, verify_bundle, load_host_model
from ego import EgoGraphs
from relational_sage import RelationalSAGE

LOW, HIGH, GRAPH, BASELINE = .30, .96, .95, .94016
REFERENCE_MANIFEST_SHA = '421d1855c2ae46f01b531f242aaa265a3a23b992d2969b24d534b1b088367fc5'


def atomic_json(path, value):
    path = safe(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w') as stream:
        json.dump(value, stream, indent=2, sort_keys=True)
        stream.flush(); os.fsync(stream.fileno())
    os.replace(tmp, path)


def key_for_row(x, row):
    """Exact, sorted uint32 feature columns; no lossy hash or identity fields."""
    start, stop = int(x.indptr[row]), int(x.indptr[row + 1])
    cols = x.indices[start:stop]
    if len(cols) > 1 and np.any(cols[:-1] >= cols[1:]):
        raise ValueError('Host columns are not strictly increasing')
    return cols.astype('<u4', copy=False).tobytes()


def get_shard(data, shard):
    from host_shards import decode_shard
    return decode_shard(data.path_for(shard), data.width, shard)


def build_registry(data, sampling_seed=1, neg_per_pos=30,
                   expected_fit_counts=(6046559, 201551)):
    """Full TRAIN primary registry and exact frozen undersample diagnostic."""
    full, fit = {}, {}
    full_occurrences, fit_occurrences = {}, {}
    full_rows = Counter(); fit_rows = Counter()
    rng = np.random.default_rng(sampling_seed)
    train = data.meta['splits']['train']
    keep_p = min(1., neg_per_pos * train['label_1'] / train['label_0'])
    for shard in data.meta['shards']:
        if shard['split'] != 'train':
            continue
        x, y, _ = get_shard(data, shard)
        neg = np.flatnonzero(y == 0)
        selected_neg = np.zeros(len(y), dtype=bool)
        selected_neg[neg] = rng.random(len(neg)) < keep_p
        for row, label in enumerate(y):
            label = int(label)
            key = key_for_row(x, row)
            full[key] = full.get(key, 0) | (1 << label)
            if key not in full_occurrences:
                full_occurrences[key] = [0, 0]
            full_occurrences[key][label] += 1
            full_rows[label] += 1
            if label or selected_neg[row]:
                fit[key] = fit.get(key, 0) | (1 << label)
                if key not in fit_occurrences:
                    fit_occurrences[key] = [0, 0]
                fit_occurrences[key][label] += 1
                fit_rows[label] += 1
    if [full_rows[0], full_rows[1]] != [train['label_0'], train['label_1']]:
        raise ValueError('Full TRAIN does not match manifest')
    if expected_fit_counts is not None and [fit_rows[0], fit_rows[1]] != list(expected_fit_counts):
        raise ValueError('Fit subset does not match frozen LightGBM sampling')
    return full, fit, full_rows, fit_rows, full_occurrences, fit_occurrences


def registry_summary(reg, row_counts, occurrences):
    labels = Counter(reg.values())
    collision_rows = [sum(occurrences[key][label] for key, mask in reg.items() if mask == 3)
                      for label in (0, 1)]
    return dict(rows={'benign': row_counts[0], 'malicious': row_counts[1],
                      'total': row_counts[0] + row_counts[1]},
                unique_vectors=len(reg), benign_only_vectors=labels[1],
                malicious_only_vectors=labels[2], conflicting_vectors=labels[3],
                conflicting_vector_rows={'benign': collision_rows[0],
                                         'malicious': collision_rows[1]})


def publish_registry(path, reg, occurrences):
    """Auditable TRAIN-only key/label-set registry, with no event identities."""
    path = safe(path)
    tmp = path.with_suffix('.jsonl.tmp')
    with tmp.open('w') as stream:
        for key in sorted(reg):
            labels = reg[key]
            stream.write(json.dumps(dict(vector_key=key.hex(),
                                         labels=[label for label in (0, 1)
                                                 if labels & (1 << label)],
                                         train_label_counts=occurrences[key])) + '\n')
        stream.flush(); os.fsync(stream.fileno())
    os.replace(tmp, path)


def score_map_from_audit(path, expected_model_sha):
    """Optional local scorer when LightGBM wheel is absent; never reads labels."""
    document = json.loads(Path(path).read_text())
    if document['model_sha256'] != expected_model_sha:
        raise ValueError('Local score-map checkpoint identity mismatch')
    return {bytes.fromhex(k): float(v) for k, v in document['scores'].items()}


def local_score_map(census_path, model_path, output):
    """Strip audited checkpoint scores from a pre-existing vector census."""
    groups = json.loads(Path(census_path).read_text())
    scores = {}
    for group in groups:
        key = np.asarray(group['columns'], dtype='<u4').tobytes().hex()
        value = float(group['scores']['lgbm'])
        if key in scores and scores[key] != value:
            raise ValueError('Census score collision')
        scores[key] = value
    atomic_json(output, dict(model_sha256=sha(model_path), scores=scores,
                             provenance=str(census_path),
                             policy='Score-only projection; no census labels used'))


def route_predictions(ml, graph, collision):
    """Collision override precedes both legacy direct branches."""
    band = (ml >= LOW) & (ml < HIGH)
    routed = collision | band
    if not np.all(np.isfinite(graph[routed])):
        raise ValueError('Required GraphSAGE score missing')
    legacy = np.where(band, graph >= GRAPH, ml >= HIGH)
    aware = np.where(routed, graph >= GRAPH, ml >= HIGH)
    return legacy, aware, band, routed


def score_distribution(values):
    if not len(values):
        return dict(count=0)
    return dict(count=int(len(values)), min=float(np.min(values)),
                p10=float(np.quantile(values, .1)), p50=float(np.median(values)),
                p90=float(np.quantile(values, .9)), max=float(np.max(values)))


def run(manifest_path, bundle, output, *, score_map=None, legacy_result=None,
        volume_root_ssd=None):
    from host_shards import HostShardDataset
    manifest_path, bundle, output = map(lambda p: Path(p).resolve(),
                                         (manifest_path, bundle, output))
    safe(output)
    if output.is_relative_to(bundle) or output.is_relative_to(manifest_path.parent):
        raise ValueError('Experiment output overlaps an input')
    output.mkdir(parents=True, exist_ok=True)
    bundle_meta = verify_bundle(bundle)
    if (bundle_meta.get('schema') != 5 or bundle_meta.get('context_policy', {}).get('timestamp_quantum_us') != 1
            or bundle_meta.get('context_policy', {}).get('checkpoint_training_quantum_us') != 1):
        raise ValueError('Full exact-time retrained GraphSAGE bundle required')
    volume_roots = {'ssd': volume_root_ssd} if volume_root_ssd is not None else None
    data = HostShardDataset(manifest_path, volume_roots=volume_roots)
    atomic_json(output / 'source_index.json',
                {str(source['source_id']): source['source']['path']
                 for source in data.meta['sources']})
    baseline_meta = json.loads((bundle / 'models/lgbm_metrics.json').read_text())
    if baseline_meta['validation']['threshold'] != BASELINE:
        raise ValueError('Frozen LightGBM threshold changed')
    for filename, field in [('vocab.json', 'vocabulary_sha256'),
                            ('split_manifest.json', 'split_sha256')]:
        if sha(manifest_path.parent / 'assets' / filename) != baseline_meta[field]:
            raise ValueError('Frozen feature/split asset mismatch: ' + filename)
    identity = dict(manifest_sha256=sha(manifest_path), bundle_sha256=sha(bundle / 'bundle_manifest.json'),
                    graph_sha256=bundle_meta['files']['provenance/graph.sqlite']['sha256'],
                    graph_model_sha256=bundle_meta['files']['provenance/model_numpy.npz']['sha256'],
                    host_model_sha256=bundle_meta['files']['models/lgbm_model.txt']['sha256'],
                    schema_sha256=sha(manifest_path.parent / 'assets/vocab.json'),
                    feature_width=data.width, key_method='exact sorted uint32 column bytes, little-endian',
                    low=LOW, high=HIGH, graph_threshold=GRAPH, baseline_threshold=BASELINE)
    if identity['manifest_sha256'] != REFERENCE_MANIFEST_SHA:
        raise ValueError('Host TEST manifest SHA-256 differs from frozen seed02 Host split')
    config = output / 'config.json'
    if config.exists() and json.loads(config.read_text()) != identity:
        raise ValueError('Resume identity mismatch; use a fresh output directory')
    atomic_json(config, identity)
    if legacy_result is not None:
        raise ValueError('Legacy score/result artifacts use a different GraphSAGE checkpoint; score this bundle end to end')
    legacy_validation = None
    full, fit, full_rows, fit_rows, full_occ, fit_occ = build_registry(data)
    if any(key not in full or fit_labels & ~full[key] for key, fit_labels in fit.items()):
        raise ValueError('Fit-only labels are not a subset of full TRAIN labels')
    collisions = {key for key, labels in full.items() if labels == 3}
    reg_summary = dict(primary=registry_summary(full, full_rows, full_occ),
                       fit_only_diagnostic=registry_summary(fit, fit_rows, fit_occ),
                       identity=identity, sampling=dict(seed=1, neg_per_pos=30,
                                                        method='frozen random_negative_undersampling',
                                                        numpy_version=np.__version__,
                                                        fit_subset_status='reconstructed from published seed and sampling code; row-identity receipt was not exported'),
                       fit_only_conflicts_not_in_primary=sum(labels == 3 and full[key] != 3
                                                             for key, labels in fit.items()),
                       full_conflicts_absent_from_fit=sum(key not in fit or fit[key] != 3
                                                          for key in collisions))
    atomic_json(output / 'collision_registry_summary.json', reg_summary)
    publish_registry(output / 'collision_registry.jsonl', full, full_occ)
    if score_map is None:
        predict_host, version = load_host_model(bundle, 'lgbm')
        trained_version = bundle_meta['model_identities']['lgbm']['library_version']
        if version != trained_version:
            raise RuntimeError(f'Frozen LightGBM requires {trained_version}; found {version}')
        def host_scores(x):
            return np.asarray(predict_host(x), dtype=np.float64)
    else:
        mapping = score_map_from_audit(score_map, identity['host_model_sha256'])
        version = 'audited frozen score lookup; model checkpoint hash verified'
        def host_scores(x):
            return np.asarray([mapping[key_for_row(x, i)] for i in range(x.shape[0])], dtype=np.float64)
    model = RelationalSAGE.load(bundle / 'provenance/model_numpy.npz')
    graph_path = bundle / 'provenance/graph.sqlite'
    db = sqlite3.connect(graph_path.as_uri() + '?mode=ro', uri=True)
    db.execute('PRAGMA cache_size=-262144')
    ego = EgoGraphs(db)
    validation_reference = json.loads((output / 'validation_metrics.json').read_text()) if (output / 'validation_metrics.json').exists() else None
    results = {}
    try:
        for split in ('val', 'test'):
            if split == 'test' and validation_reference is None:
                validation_reference = results['val']
                atomic_json(output / 'validation_metrics.json', validation_reference)
            if split == 'test':
                # Fixed routing and threshold; TEST never changes the policy.
                assert validation_reference['policy']['graph_threshold'] == GRAPH
            totals = defaultdict(lambda: np.zeros(4, dtype=np.int64))
            routing = Counter(); transitions = Counter(); scores_by_label = {0: [], 1: []}
            errors_file = output / 'test_errors_full_scores.jsonl'
            changes_file = output / 'event_transition_audit.jsonl'
            if split == 'test':
                errors_tmp = errors_file.with_suffix('.jsonl.tmp')
                changes_tmp = changes_file.with_suffix('.jsonl.tmp')
                errors_out = errors_tmp.open('w')
                changes_out = changes_tmp.open('w')
            try:
                for index, shard in enumerate(data.meta['shards']):
                    if shard['split'] != split:
                        continue
                    x, y, lines = get_shard(data, shard)
                    ml = host_scores(x)
                    collision = np.fromiter((key_for_row(x, i) in collisions for i in range(len(y))),
                                            dtype=bool, count=len(y))
                    band = (ml >= LOW) & (ml < HIGH)
                    routed = collision | band
                    extra = collision & ~band
                    cache = output / f'{split}-graph-{index:04d}.npz'
                    receipt = cache.with_suffix('.json')
                    cached_receipt = json.loads(receipt.read_text()) if receipt.exists() else None
                    if (cache.exists() and cached_receipt is not None
                            and cached_receipt.get('identity') == identity
                            and cached_receipt.get('source_id') == shard['source_id']
                            and cached_receipt.get('shard_sha256') == shard['sha256']
                            and sha(cache) == cached_receipt['sha256']):
                        with np.load(cache, allow_pickle=False) as f:
                            graph = f['graph']
                        if graph.shape != ml.shape or not np.all(np.isfinite(graph[routed])):
                            raise ValueError('Invalid graph score cache')
                    else:
                        graph = np.full(len(y), np.nan, dtype=np.float64)
                        missing = []
                        for row in np.flatnonzero(routed):
                            target = db.execute('SELECT host,proc,t FROM target WHERE source=? AND line=?',
                                                (shard['source_id'], int(lines[row]))).fetchone()
                            if target is None or not target[0] or not target[1] or target[2] is None:
                                missing.append(dict(source_id=shard['source_id'], source_line=int(lines[row]),
                                                    reason='missing_target_or_identity'))
                                continue
                            graph[row] = model.predict(ego.build(*target))['probability']
                        if missing:
                            atomic_json(output / f'{split}-missing-{index:04d}.json', missing)
                            raise ValueError(f'{len(missing)} required graph scores missing in shard {index}')
                        tmp = cache.with_suffix('.npz.tmp')
                        with tmp.open('wb') as f:
                            np.savez_compressed(f, graph=graph)
                            f.flush(); os.fsync(f.fileno())
                        os.replace(tmp, cache)
                        atomic_json(receipt, dict(sha256=sha(cache), source_id=shard['source_id'],
                                                  shard_sha256=shard['sha256'], identity=identity))
                    old, new, checked_band, checked_routed = route_predictions(ml, graph, collision)
                    if not np.array_equal(band, checked_band) or not np.array_equal(routed, checked_routed):
                        raise ValueError('Routing mask disagreement')
                    route_reason = np.where(collision, 'collision', np.where(band, 'legacy_graph', 'legacy_direct'))
                    totals['legacy'] += counts(y, old)
                    totals['collision_aware'] += counts(y, new)
                    for tag, mask in [('collision', collision), ('noncollision', ~collision)]:
                        totals['legacy_' + tag] += counts(y[mask], old[mask])
                        totals['new_' + tag] += counts(y[mask], new[mask])
                    routing['rows'] += len(y)
                    routing['collision'] += int(collision.sum())
                    routing['collision_low'] += int((collision & (ml < LOW)).sum())
                    routing['collision_middle'] += int((collision & band).sum())
                    routing['collision_high'] += int((collision & (ml >= HIGH)).sum())
                    routing['newly_routed'] += int(extra.sum())
                    routing['legacy_graph'] += int(band.sum())
                    routing['new_graph'] += int(routed.sum())
                    routing['collision_corrected'] += int((collision & (old != y) & (new == y)).sum())
                    routing['collision_new_errors'] += int((collision & (old == y) & (new != y)).sum())
                    for label in (0, 1):
                        scores_by_label[label].extend(graph[collision & (y == label)].tolist())
                    changed = np.flatnonzero(old != new)
                    if len(changed) and not np.all(collision[changed] & ~band[changed]):
                        raise ValueError('Non-collision or legacy-band prediction changed')
                    transitions.update(f'{"FP" if y[i] == 0 else "FN"}_' +
                                       ('corrected' if new[i] == y[i] else 'introduced') for i in changed)
                    if split == 'test':
                        for row in np.flatnonzero(new != y):
                            item = dict(source_id=shard['source_id'], source_line=int(lines[row]),
                                        shard=shard['path'], label=int(y[row]),
                                        ml_score=float(ml[row]),
                                        graph_score=float(graph[row]) if np.isfinite(graph[row]) else None,
                                        route='graph' if routed[row] else 'direct',
                                        route_reason=str(route_reason[row]), collision=bool(collision[row]),
                                        threshold=(GRAPH if routed[row] else
                                                   LOW if ml[row] < LOW else HIGH),
                                        prediction=int(new[row]), legacy_prediction=int(old[row]),
                                        baseline_prediction=int(ml[row] >= BASELINE),
                                        graph_score_used_for_decision=bool(routed[row]),
                                        error='FP' if y[row] == 0 else 'FN')
                            errors_out.write(json.dumps(item) + '\n')
                        for row in changed:
                            item = dict(source_id=shard['source_id'], source_line=int(lines[row]),
                                        label=int(y[row]), ml_score=float(ml[row]),
                                        graph_score=float(graph[row]), collision=bool(collision[row]),
                                        old_prediction=int(old[row]), new_prediction=int(new[row]),
                                        old_route='direct', new_route='graph',
                                        reason='collision_override_' + ('low' if ml[row] < LOW else 'high'),
                                        old_error=bool(old[row] != y[row]), new_error=bool(new[row] != y[row]))
                            changes_out.write(json.dumps(item) + '\n')
                    print(f'{split} shard {index}: rows={len(y)} collision={int(collision.sum())} '
                          f'extra={int(extra.sum())} transitions={len(changed)}', flush=True)
            finally:
                if split == 'test':
                    errors_out.close(); changes_out.close()
            if split == 'test':
                os.replace(errors_tmp, errors_file)
                os.replace(changes_tmp, changes_file)
            routing['collision_percent'] = 100 * routing['collision'] / max(1, routing['rows'])
            routing['newly_routed_percent_of_rows'] = 100 * routing['newly_routed'] / max(1, routing['rows'])
            result = dict(identity=identity, policy=dict(collision_override=True, low=LOW, high=HIGH,
                                                         graph_threshold=GRAPH, selection='fixed a priori'),
                          scorer=version, routing=dict(routing), transition_counts=dict(transitions),
                          legacy=metrics(totals['legacy']), collision_aware=metrics(totals['collision_aware']),
                          groups={tag: dict(legacy=metrics(totals['legacy_' + tag]),
                                            collision_aware=metrics(totals['new_' + tag]))
                                  for tag in ('collision', 'noncollision')},
                          collision_graph_score_by_true_label={str(k): score_distribution(np.asarray(v))
                                                                for k, v in scores_by_label.items()})
            if result['groups']['noncollision']['legacy'] != result['groups']['noncollision']['collision_aware']:
                raise ValueError('Non-collision group changed')
            old_c = result['legacy']['confusion']
            new_c = result['collision_aware']['confusion']
            expected_new = dict(
                tp=old_c['tp'] + transitions['FN_corrected'] - transitions['FN_introduced'],
                fp=old_c['fp'] - transitions['FP_corrected'] + transitions['FP_introduced'],
                fn=old_c['fn'] - transitions['FN_corrected'] + transitions['FN_introduced'],
                tn=old_c['tn'] + transitions['FP_corrected'] - transitions['FP_introduced'])
            if new_c != expected_new:
                raise ValueError('Event transitions do not reconcile with confusion matrices')
            if split == 'test':
                if result['legacy']['rows'] != data.meta['splits']['test']['rows']:
                    raise ValueError('TEST row count mismatch')
                result['validation_selection'] = validation_reference['validation_selection']
            else:
                if result['legacy']['rows'] != data.meta['splits']['val']['rows']:
                    raise ValueError('Validation row count mismatch')
                if legacy_validation is not None and result['legacy']['confusion'] != legacy_validation:
                    raise ValueError('New-checkpoint validation reference mismatch')
                result['validation_selection'] = dict(
                    preferred_policy=('collision_aware' if result['collision_aware']['f1'] > result['legacy']['f1']
                                      else 'legacy'),
                    criterion='higher validation F1; legacy wins ties',
                    test_evaluated_policy='predeclared fixed collision-aware candidate')
            results[split] = result
            atomic_json(output / ('validation_metrics.json' if split == 'val' else 'test_metrics.json'), result)
            if split == 'val':
                validation_reference = result
        summary = output / 'summary.md'
        v, t = results['val'], results['test']
        def metric_line(name, value):
            old, new = value['legacy'], value['collision_aware']
            old_c, new_c = old['confusion'], new['confusion']
            return (f"| {name} | {old_c['tp']:,}/{old_c['fp']:,}/{old_c['fn']:,}/{old_c['tn']:,} | "
                    f"{new_c['tp']:,}/{new_c['fp']:,}/{new_c['fn']:,}/{new_c['tn']:,} | "
                    f"{old['precision']:.6f}→{new['precision']:.6f} | "
                    f"{old['recall']:.6f}→{new['recall']:.6f} | "
                    f"{old['f1']:.6f}→{new['f1']:.6f} | "
                    f"{old['fpr']:.6f}→{new['fpr']:.6f} |")
        lines = [
            '# LightGBM + exact-time GraphSAGE collision-aware routing', '',
            'The only policy change is routing every exact full-TRAIN label-conflicting '
            'Host vector to the retrained exact-time GraphSAGE at threshold 0.95. Non-collision '
            'events retain the legacy 0.30/0.96 gates. The LightGBM model, Host feature '
            'schema, split and thresholds are frozen; the GraphSAGE checkpoint is new.', '',
            f"Manifest SHA-256: `{identity['manifest_sha256']}` (frozen seed02 Host split). "
            f"LightGBM: `{identity['host_model_sha256']}`; GraphSAGE: "
            f"`{identity['graph_model_sha256']}`. The full TRAIN partition has "
            f"{reg_summary['primary']['rows']['total']:,} rows and "
            f"{reg_summary['primary']['unique_vectors']:,} exact vectors; "
            f"{reg_summary['primary']['conflicting_vectors']:,} conflict. "
            f"The fit-only diagnostic excludes 333 benign rows but still has "
            f"{reg_summary['fit_only_diagnostic']['conflicting_vectors']:,} conflicting vectors.", '',
            '| Split | Legacy TP/FP/FN/TN | Collision-aware TP/FP/FN/TN | Precision | Recall | F1 | FPR |',
            '| --- | --- | --- | --- | --- | --- | --- |',
            metric_line('Validation', v), metric_line('TEST', t), '',
        ]
        for name, value in [('Validation', v), ('TEST', t)]:
            r, tr = value['routing'], value['transition_counts']
            lines += [
                f"## {name}", '',
                f"Collision events: {r['collision']:,}/{r['rows']:,} "
                f"({100*r['collision']/r['rows']:.2f}%); ML bands <0.30 / 0.30–0.96 / ≥0.96: "
                f"{r['collision_low']:,} / {r['collision_middle']:,} / {r['collision_high']:,}. "
                f"GraphSAGE calls: {r['legacy_graph']:,}→{r['new_graph']:,}; "
                f"+{r['newly_routed']:,} ({100*r['newly_routed']/max(1,r['legacy_graph']):.1f}% "
                'over legacy workload).', '',
                f"Collision-group errors corrected: {r['collision_corrected']:,}; "
                f"new errors introduced: {r['collision_new_errors']:,}. "
                f"Transitions FP corrected/new: {tr.get('FP_corrected',0):,}/"
                f"{tr.get('FP_introduced',0):,}; FN corrected/new: "
                f"{tr.get('FN_corrected',0):,}/{tr.get('FN_introduced',0):,}.", '',
                f"Low-ML collision override rescues {tr.get('FN_corrected',0):,} FN but "
                f"introduces {tr.get('FP_introduced',0):,} FP. High-ML collision "
                f"override vetoes {tr.get('FP_corrected',0):,} FP but introduces "
                f"{tr.get('FN_introduced',0):,} FN.", '',
                f"Collision-group F1: {value['groups']['collision']['legacy']['f1']:.6f}→"
                f"{value['groups']['collision']['collision_aware']['f1']:.6f}; "
                f"non-collision F1: {value['groups']['noncollision']['legacy']['f1']:.6f} "
                '(identical by invariant).', '',
            ]
        recommendation = ('The fixed routing rule improved validation F1; retain it as an '
                          'experimental candidate, with TEST used only for frozen evaluation.'
                          if v['collision_aware']['f1'] > v['legacy']['f1'] else
                          'The fixed routing rule did not improve validation F1; do not '
                          'promote it as the default cascade policy.')
        lines += ['## Interpretation', '', recommendation,
                  'Legacy and collision-aware TEST predictions use the same new GraphSAGE checkpoint. '
                  'Every changed prediction is keyed by `(source_id, source_line)` in '
                  '`event_transition_audit.jsonl`, and every final FP/FN is in '
                  '`test_errors_full_scores.jsonl`. Confusion deltas reconcile exactly '
                  'with the transition counts. TEST labels were never used for registry '
                  'construction, routing, or threshold selection.', '',
                  f"The conflicting TRAIN vectors contain "
                  f"{reg_summary['primary']['conflicting_vector_rows']['benign']:,} benign "
                  f"and {reg_summary['primary']['conflicting_vector_rows']['malicious']:,} "
                  'malicious rows. Collision membership is therefore common among '
                  'benign events; unconditional routing exposes its high-score tail '
                  'to GraphSAGE false positives.', '',
                  'Collision membership proves only that the Host vector is ambiguous '
                  'within TRAIN. GraphSAGE can still make new errors; its existing '
                  'Process/Host graph lacks FILE/MODULE nodes and was trained with '
                  'one-second snapshots. The local score-only cache was stripped '
                  'from a frozen-checkpoint census because the LightGBM wheel is '
                  'unavailable locally; the Kaggle notebook uses the native frozen '
                  'LightGBM checkpoint. Both runs verify the same model hash. '
                  'The fit-only registry is reconstructed from the published '
                  'sampling algorithm and seed; the downloaded model result did '
                  'not include row-identity receipts for those 333 excluded benign '
                  'TRAIN rows.', '']
        summary_tmp = summary.with_suffix('.md.tmp')
        with summary_tmp.open('w') as stream:
            stream.write('\n'.join(lines))
            stream.flush(); os.fsync(stream.fileno())
        os.replace(summary_tmp, summary)
        return results
    finally:
        db.close()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--manifest', required=True)
    parser.add_argument('--bundle', required=True)
    parser.add_argument('--output', required=True)
    parser.add_argument('--score-map')
    parser.add_argument('--legacy-result')
    parser.add_argument('--volume-root-ssd')
    args = parser.parse_args()
    run(args.manifest, args.bundle, args.output,
        score_map=args.score_map, legacy_result=args.legacy_result,
        volume_root_ssd=args.volume_root_ssd)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/conditional_collision_runtime.py
"""Validation-selected conditional collision routing with exact-time retrained GraphSAGE.

Low-band extra FP and high-band extra FN budgets are fixed before TEST.
The middle band and every non-collision decision retain the legacy gates, scored by this exact-time checkpoint.
"""
from collections import Counter, defaultdict, deque
from concurrent.futures import ProcessPoolExecutor
import multiprocessing
import argparse
import hashlib
import json
import os
from pathlib import Path
import sqlite3
import sys

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[2] / 'preprocess'))

from collision_aware_runtime import (BASELINE, GRAPH, HIGH, LOW, REFERENCE_MANIFEST_SHA,
    atomic_json, build_registry, get_shard, key_for_row,
    publish_registry, registry_summary, score_distribution, score_map_from_audit)
from cascade_runtime import counts, metrics, safe, sha, verify_bundle, load_host_model
from ego import EgoGraphs
from relational_sage import RelationalSAGE

LOW_FP_BUDGET = 10
HIGH_FN_BUDGET = 10
MIN_F1_GAIN = 0.0001
GRAPH_BATCH_SIZE = 256
GRAPH_BUILD_BATCH_SIZE = 64
try:
    _VISIBLE_CPUS = len(os.sched_getaffinity(0))
except (AttributeError, OSError):
    _VISIBLE_CPUS = os.cpu_count() or 2
GRAPH_CPU_WORKERS = max(1, min(8, _VISIBLE_CPUS - 1))
RELATION_NAMES = ('creates', 'communicates', 'communicates_reverse', 'hosts')
INFERENCE_VERSION = 'dense_disjoint_graph_batch_fp64_v1'


class DenseCudaGraphSAGE:
    """Batch disjoint ego graphs across every visible CUDA GPU."""
    def __init__(self, model, batch_size=GRAPH_BATCH_SIZE):
        self.model = model
        self.batch_size = int(batch_size)
        try:
            import torch
        except ImportError:
            self.torch = None
            self.network = None
            self.device_ids = []
            return
        if not torch.cuda.is_available():
            self.torch = None
            self.network = None
            self.device_ids = []
            return
        self.torch = torch
        self.device_ids = list(range(torch.cuda.device_count()))
        torch.backends.cuda.matmul.allow_tf32 = False

        class BatchNetwork(torch.nn.Module):
            def __init__(self, params):
                super().__init__()
                for name, value in params.items():
                    self.register_buffer('p_' + name, torch.as_tensor(value, dtype=torch.float64))

            def forward(self, x, adjacency, roots, hosts):
                import torch
                h = x
                for layer in (1, 2):
                    z = h @ getattr(self, f'p_W{layer}_self') + getattr(self, f'p_b{layer}')
                    for rid, relation in enumerate(RELATION_NAMES):
                        neighborhood = torch.bmm(adjacency[:, rid], h)
                        z = z + neighborhood @ getattr(self, f'p_W{layer}_{relation}')
                    h = torch.relu(z)
                row = torch.arange(len(roots), device=x.device)
                readout = torch.cat((h[row, roots], h[row, hosts]), dim=1)
                return torch.sigmoid(readout @ self.p_head + self.p_bias[0])

        network = BatchNetwork(model.params).to('cuda:0')
        if len(self.device_ids) > 1:
            self.network = torch.nn.DataParallel(network, device_ids=self.device_ids, output_device=0)
        else:
            self.network = network
        self.verified = False

    @property
    def enabled(self):
        return self.torch is not None

    def _pack(self, graphs):
        batch = len(graphs)
        max_nodes = max(len(g['x']) for g in graphs)
        dim = self.model.input_dim
        x = np.zeros((batch, max_nodes, dim), dtype=np.float64)
        adjacency = np.zeros((batch, len(RELATION_NAMES), max_nodes, max_nodes), dtype=np.float64)
        roots = np.zeros(batch, dtype=np.int64)
        hosts = np.zeros(batch, dtype=np.int64)
        relation_index = {name: i for i, name in enumerate(RELATION_NAMES)}
        for b, graph in enumerate(graphs):
            n = len(graph['x'])
            x[b, :n] = np.asarray(graph['x'], dtype=np.float64)
            roots[b] = int(graph['root'])
            hosts[b] = int(graph['host_root'])
            for src, dst, relation in graph['edges']:
                if relation not in relation_index or src < 0 or dst < 0 or src >= n or dst >= n:
                    raise ValueError('Invalid edge in GraphSAGE batch')
                adjacency[b, relation_index[relation], dst, src] = 1.0
            degree = adjacency[b].sum(axis=2, keepdims=True)
            adjacency[b] /= np.maximum(degree, 1.0)
        return x, adjacency, roots, hosts

    def _score_batch(self, graphs):
        torch = self.torch
        x, adjacency, roots, hosts = self._pack(graphs)
        x = torch.as_tensor(x, device='cuda:0')
        adjacency = torch.as_tensor(adjacency, device='cuda:0')
        roots = torch.as_tensor(roots, device='cuda:0')
        hosts = torch.as_tensor(hosts, device='cuda:0')
        probability = self.network(x, adjacency, roots, hosts).detach().cpu().numpy().astype(np.float64, copy=False)
        if not self.verified:
            reference = np.asarray([self.model.predict(g)['probability'] for g in graphs[:min(8, len(graphs))]])
            error = float(np.max(np.abs(probability[:len(reference)] - reference))) if len(reference) else 0.0
            if error > 2e-8:
                raise ValueError(f'CUDA GraphSAGE batch differs from NumPy reference: max_abs_error={error}')
            self.verified = True
            print(f'CUDA batch verified against NumPy; devices={self.device_ids}; max abs error={error:.3g}', flush=True)
        return probability

    def score(self, graphs):
        if not self.enabled:
            return np.asarray([self.model.predict(g)['probability'] for g in graphs], dtype=np.float64)
        scores = []
        for start in range(0, len(graphs), self.batch_size):
            scores.extend(self._score_batch(graphs[start:start + self.batch_size]))
        return np.asarray(scores, dtype=np.float64)


_GRAPH_WORKER_DB = None
_GRAPH_WORKER_EGO = None

def _init_graph_worker(db_path):
    global _GRAPH_WORKER_DB, _GRAPH_WORKER_EGO
    db_uri = Path(db_path).resolve().as_uri() + '?mode=ro'
    _GRAPH_WORKER_DB = sqlite3.connect(db_uri, uri=True)
    _GRAPH_WORKER_DB.execute('PRAGMA cache_size=-65536')
    _GRAPH_WORKER_DB.execute('PRAGMA mmap_size=1073741824')
    _GRAPH_WORKER_EGO = EgoGraphs(_GRAPH_WORKER_DB)

def _build_graph_chunk(items):
    if _GRAPH_WORKER_EGO is None:
        raise RuntimeError('Graph worker was not initialized')
    rows = [int(row) for row, _ in items]
    graphs = [_GRAPH_WORKER_EGO.build(*target) for _, target in items]
    return rows, graphs

def atomic_npz(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('wb') as stream:
        np.savez(stream, **arrays)
        stream.flush(); os.fsync(stream.fileno())
    os.replace(tmp, path)


def policy_fingerprint(policy):
    payload = json.dumps(policy, sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(payload).hexdigest()


def checkpoint_identity(identity):
    # Numerical outputs do not depend on device count or CPU worker scheduling.
    return {key:value for key,value in identity.items()
            if key not in {'graph_inference_backend','graph_batch_size',
                           'graph_build_batch_size','graph_cpu_workers'}}


def registry_identity_matches(saved, current):
    # Collision keys depend only on the frozen Host rows, labels and schema.
    required = ('manifest_sha256','schema_sha256','feature_width','key_method')
    return all(saved.get(key) == current.get(key) for key in required)


def inference_backend_name():
    try:
        import torch
        return f'cuda{torch.cuda.device_count()}x_' + INFERENCE_VERSION if torch.cuda.is_available() else 'numpy_cpu_reference'
    except ImportError:
        return 'numpy_cpu_reference'


def threshold_options(labels, scores, *, band):
    """Pareto points under the fixed error budget; inclusive graph threshold."""
    labels = np.asarray(labels, dtype=np.uint8)
    scores = np.asarray(scores, dtype=np.float64)
    if not np.all(np.isfinite(scores)):
        raise ValueError('Non-finite collision graph score')
    # At 1.0, all finite model probabilities are below threshold.
    values = np.unique(np.concatenate(([1.0], scores)))
    order = np.argsort(scores, kind='stable')[::-1]
    ranked = scores[order]
    positive = np.cumsum(labels[order] == 1)
    negative = np.cumsum(labels[order] == 0)
    best = {}
    for threshold in values:
        selected = int(np.searchsorted(-ranked, -threshold, side='right'))
        tp = int(positive[selected-1]) if selected else 0
        fp = int(negative[selected-1]) if selected else 0
        if band == 'low':
            if fp > LOW_FP_BUDGET:
                continue
            budget_used, benefit = fp, tp
        else:
            fn = int(np.sum(labels == 1)) - tp
            corrected_fp = int(np.sum(labels == 0)) - fp
            if fn > HIGH_FN_BUDGET:
                continue
            budget_used, benefit = fn, corrected_fp
        incumbent = best.get(budget_used)
        if incumbent is None or benefit > incumbent['benefit'] or (
                benefit == incumbent['benefit'] and threshold > incumbent['threshold']):
            best[budget_used] = dict(threshold=float(threshold), budget_used=budget_used,
                                     benefit=benefit, tp=tp, fp=fp,
                                     fn=int(np.sum(labels == 1))-tp,
                                     tn=int(np.sum(labels == 0))-fp)
    return list(best.values())


def select_policy(legacy_counts, low, high):
    """Pick validation-only constrained F1 optimum; legacy wins unclear gains."""
    old = metrics(np.asarray(legacy_counts, dtype=np.int64))
    candidates = []
    for left in threshold_options(*low, band='low'):
        for right in threshold_options(*high, band='high'):
            delta = np.array([left['tp']-right['fn'], left['fp']-right['tn'],
                              -left['tp']+right['fn'], -left['fp']+right['tn']], dtype=np.int64)
            candidate = metrics(np.asarray(legacy_counts, dtype=np.int64)+delta)
            candidates.append((candidate['f1'], left, right, candidate))
    if not candidates:
        raise ValueError('No budget-compliant candidates')
    candidates.sort(key=lambda item: (item[0], -item[3]['confusion']['fp'],
                                      -item[3]['confusion']['fn']), reverse=True)
    f1, left, right, measured = candidates[0]
    accepted = f1 >= old['f1'] + MIN_F1_GAIN
    return dict(accepted=accepted, selected='conditional' if accepted else 'legacy',
                low_threshold=left['threshold'], high_threshold=right['threshold'],
                low_extra_fp=left['fp'], low_rescued_fn=left['tp'],
                high_vetoed_fp=right['tn'], high_extra_fn=right['fn'],
                validation_candidate=measured, validation_legacy=old,
                fp_budget_low=LOW_FP_BUDGET, fn_budget_high=HIGH_FN_BUDGET,
                minimum_f1_gain=MIN_F1_GAIN, candidates_evaluated=len(candidates),
                selection_basis='validation labels only; fixed budgets; middle and noncollision unchanged')


def predict(ml, graph, collision, policy):
    middle = (ml >= LOW) & (ml < HIGH)
    old = np.where(middle, graph >= GRAPH, ml >= HIGH)
    if not policy['accepted']:
        return old, old.copy(), np.full(len(ml), 'legacy_frozen', dtype=object)
    low = collision & (ml < LOW)
    high = collision & (ml >= HIGH)
    required = middle | low | high
    if not np.all(np.isfinite(graph[required])):
        raise ValueError('Required GraphSAGE score missing')
    new = old.copy()
    new[low] = graph[low] >= policy['low_threshold']
    new[high] = graph[high] >= policy['high_threshold']
    reason = np.where(low, 'collision_low', np.where(high, 'collision_high_veto',
                      np.where(middle, 'legacy_middle', 'legacy_direct')))
    return old, new, reason


def per_vector_audit(registry, val_rows, policy, output):
    groups = defaultdict(lambda: [[], []])
    for row in val_rows:
        groups[row['key']][row['label']].append(row)
    identical = defaultdict(list)
    candidate_pairs = []
    for key, (benign, malignant) in groups.items():
        by_score = defaultdict(lambda: [[], []])
        for row in benign+malignant:
            by_score[row['graph']][row['label']].append(row)
        for score, (b, m) in by_score.items():
            if b and m:
                candidate_pairs.append((key, score, b[0], m[0]))
    # Equal scores alone do not prove equal graph input. Verify encoded tensors.
    db_path = output['bundle'] / 'provenance/graph.sqlite'
    db = sqlite3.connect(db_path.as_uri() + '?mode=ro', uri=True)
    ego = EgoGraphs(db)
    checked = 0
    try:
        for key, score, benign, malignant in candidate_pairs:
            if checked >= 2000:
                break
            digests = []
            for row in (benign, malignant):
                target = db.execute('SELECT host,proc,t,split FROM target WHERE source=? AND line=?',
                                    (row['source_id'], row['source_line'])).fetchone()
                if target is None or target[3] != 2:
                    raise ValueError('Validation graph target missing')
                graph = ego.build(*target[:3])
                content = graph['x'].tobytes() + repr(graph['edges']).encode()
                digests.append(hashlib.sha256(content).hexdigest())
            checked += 1
            if digests[0] == digests[1]:
                identical[key].append(dict(graph_score=score, graph_encoding_sha256=digests[0],
                                           benign_event=[benign['source_id'], benign['source_line']],
                                           malicious_event=[malignant['source_id'], malignant['source_line']],
                                           interpretation='representation_limitation'))
    finally:
        db.close()
    audit_path = output['path'] / 'validation_per_vector_audit.jsonl'
    with audit_path.open('w') as stream:
        for key in sorted(registry):
            item = registry[key]
            if item['labels'] != [0, 1]:
                continue
            benign, malignant = groups[key]
            conf = Counter()
            for row in benign+malignant:
                old = row['legacy_prediction']
                new = row['candidate_prediction'] if policy['accepted'] else old
                if new != row['label']:
                    conf['fp' if row['label'] == 0 else 'fn'] += 1
                if old != row['label']:
                    conf['legacy_fp' if row['label'] == 0 else 'legacy_fn'] += 1
            example = (benign or malignant)
            ml_score = example[0]['ml'] if example else None
            ml_band = ('low' if ml_score < LOW else 'middle' if ml_score < HIGH else 'high') if ml_score is not None else None
            stream.write(json.dumps(dict(vector_key=key.hex(),
                train_label_counts=item['train_label_counts'],
                validation_ml_score=ml_score, validation_ml_band=ml_band,
                validation_counts=dict(benign=len(benign), malicious=len(malignant)),
                validation_scores=dict(benign=score_distribution(np.array([r['graph'] for r in benign])),
                                       malicious=score_distribution(np.array([r['graph'] for r in malignant]))),
                validation_errors={name:int(conf[name]) for name in ('legacy_fp','legacy_fn','fp','fn')},
                verified_identical_graph_input_examples=identical[key][:5],
                representation_limitation_verified=bool(identical[key]))) + '\n')
    atomic_json(output['path'] / 'representation_limitations.json', dict(
        equal_score_opposite_label_groups=len(candidate_pairs), groups_checked=checked,
        check_cap=2000, verified_pairs=sum(len(v) for v in identical.values()),
        vectors_with_verified_limitations=sum(bool(v) for v in identical.values()),
        scope='one representative opposite-label pair per vector and identical score; not exhaustive'))


def run(manifest_path, bundle, output, *, score_map=None, cache_source=None,
        registry_source=None, volume_root_ssd=None):
    from host_shards import HostShardDataset
    manifest_path, bundle, output = map(lambda p: Path(p).resolve(),
                                         (manifest_path, bundle, output))
    safe(output)
    if output.is_relative_to(bundle) or output.is_relative_to(manifest_path.parent):
        raise ValueError('Output overlaps input')
    output.mkdir(parents=True, exist_ok=True)
    bundle_meta = verify_bundle(bundle)
    if (bundle_meta.get('schema') != 5 or bundle_meta.get('context_policy', {}).get('timestamp_quantum_us') != 1
            or bundle_meta.get('context_policy', {}).get('checkpoint_training_quantum_us') != 1):
        raise ValueError('Full exact-time retrained GraphSAGE bundle required')
    data = HostShardDataset(manifest_path,
        volume_roots={'ssd': volume_root_ssd} if volume_root_ssd else None)
    identity = dict(manifest_sha256=sha(manifest_path), bundle_sha256=sha(bundle / 'bundle_manifest.json'),
                    graph_sha256=bundle_meta['files']['provenance/graph.sqlite']['sha256'],
                    graph_model_sha256=bundle_meta['files']['provenance/model_numpy.npz']['sha256'],
                    host_model_sha256=bundle_meta['files']['models/lgbm_model.txt']['sha256'],
                    schema_sha256=sha(manifest_path.parent / 'assets/vocab.json'),
                    feature_width=data.width, key_method='exact sorted uint32 column bytes, little-endian',
                    graph_inference_backend=inference_backend_name(), graph_batch_size=GRAPH_BATCH_SIZE,
                    graph_build_batch_size=GRAPH_BUILD_BATCH_SIZE, graph_cpu_workers=GRAPH_CPU_WORKERS,
                    low=LOW, high=HIGH, graph_threshold=GRAPH, baseline_threshold=BASELINE,
                    graph_inference_algorithm=INFERENCE_VERSION,
                    policy_selection=dict(low_fp_budget=LOW_FP_BUDGET, high_fn_budget=HIGH_FN_BUDGET,
                                          minimum_validation_f1_gain=MIN_F1_GAIN))
    if identity['manifest_sha256'] != REFERENCE_MANIFEST_SHA:
        raise ValueError('Frozen Host manifest mismatch')
    baseline_meta = json.loads((bundle / 'models/lgbm_metrics.json').read_text())
    if baseline_meta['validation']['threshold'] != BASELINE:
        raise ValueError('Frozen LightGBM threshold mismatch')
    for filename, field in (('vocab.json','vocabulary_sha256'),
                            ('split_manifest.json','split_sha256')):
        if sha(manifest_path.parent / 'assets' / filename) != baseline_meta[field]:
            raise ValueError('Frozen split/feature asset mismatch')
    atomic_json(output / 'config.json', identity)
    atomic_json(output / 'source_index.json',
        {str(s['source_id']): s['source']['path'] for s in data.meta['sources']})
    registry_candidate = Path(registry_source).resolve() if registry_source else output
    registry_files = [registry_candidate / name for name in
                      ('collision_registry.jsonl','collision_registry_summary.json')]
    registry_reused = all(path.is_file() for path in registry_files)
    if registry_reused:
        summary = json.loads(registry_files[1].read_text())
        saved_identity = summary.get('identity')
        if not saved_identity or not registry_identity_matches(saved_identity, identity):
            raise ValueError('TRAIN registry manifest/schema/model identity mismatch')
        registry = {}
        with registry_files[0].open() as stream:
            for line in stream:
                item = json.loads(line)
                if item.get('labels') == [0, 1]:
                    key = bytes.fromhex(item['vector_key'])
                    if key in registry:
                        raise ValueError('Duplicate key in TRAIN collision registry')
                    registry[key] = dict(labels=item['labels'],
                                         train_label_counts=item['train_label_counts'])
        print(f'Reused TRAIN collision registry: {len(registry):,} vectors from {registry_candidate}', flush=True)
    else:
        full, fit, full_rows, fit_rows, full_occ, fit_occ = build_registry(data)
        registry = {k: dict(labels=[0,1], train_label_counts=full_occ[k])
                    for k,v in full.items() if v == 3}
        summary = dict(primary=registry_summary(full, full_rows, full_occ),
                       fit_only_diagnostic=registry_summary(fit, fit_rows, fit_occ),
                       identity=identity)
    atomic_json(output / 'collision_registry_summary.json', summary)
    with (output / 'collision_registry.jsonl').open('w') as stream:
        for key,item in sorted(registry.items()):
            stream.write(json.dumps(dict(vector_key=key.hex(), **item))+'\n')
    if score_map:
        mapping = score_map_from_audit(score_map, identity['host_model_sha256'])
        def host_scores(x):
            return np.array([mapping[key_for_row(x,i)] for i in range(x.shape[0])])
        scorer = 'audited frozen score-only map'
    else:
        scorer_fn, version = load_host_model(bundle, 'lgbm')
        if version != bundle_meta['model_identities']['lgbm']['library_version']:
            raise ValueError('LightGBM library version mismatch')
        def host_scores(x):
            return np.asarray(scorer_fn(x), dtype=np.float64)
        scorer = version
    model = None
    db = None
    ego = None
    graph_pool = None
    batch_scorer = None
    def graph_scores(split, index, shard, lines, routed):
        nonlocal model, db, ego, graph_pool, batch_scorer
        if cache_source:
            path = Path(cache_source) / f'{split}-graph-{index:04d}.npz'
            receipt = json.loads(path.with_suffix('.json').read_text())
            if (receipt['identity'] != identity or receipt['source_id'] != shard['source_id']
                    or receipt['shard_sha256'] != shard['sha256'] or sha(path) != receipt['sha256']):
                raise ValueError('Graph cache receipt mismatch')
            with np.load(path, allow_pickle=False) as file:
                graph = file['graph']
            if len(graph) != len(lines) or not np.all(np.isfinite(graph[routed])):
                raise ValueError('Missing required frozen graph score')
            return graph
        if model is None:
            model = RelationalSAGE.load(bundle / 'provenance/model_numpy.npz')
            batch_scorer = DenseCudaGraphSAGE(model)
            backend = f'CUDA devices={batch_scorer.device_ids}' if batch_scorer.enabled else 'NumPy CPU'
            print(f'GraphSAGE inference: {backend}; graph CPU workers={GRAPH_CPU_WORKERS}', flush=True)
            db_path = bundle / 'provenance/graph.sqlite'
            db = sqlite3.connect(db_path.as_uri() + '?mode=ro', uri=True)
            db.execute('PRAGMA cache_size=-262144')
            db.execute('PRAGMA mmap_size=1073741824')
            if GRAPH_CPU_WORKERS > 1:
                graph_pool = ProcessPoolExecutor(max_workers=GRAPH_CPU_WORKERS,
                    mp_context=multiprocessing.get_context('spawn'), initializer=_init_graph_worker,
                    initargs=(str(db_path),))
            else:
                ego = EgoGraphs(db)
        scores = np.full(len(lines), np.nan, dtype=np.float64)
        selected = np.flatnonzero(routed)
        missing = []
        targets = {}
        # Batch target lookups to reduce Python↔SQLite round trips per event.
        for begin in range(0, len(selected), 700):
            rows = selected[begin:begin + 700]
            source_lines = [int(lines[row]) for row in rows]
            placeholders = ','.join('?' for _ in source_lines)
            query = f'SELECT line,host,proc,t FROM target WHERE source=? AND line IN ({placeholders})'
            for line, host, proc, timestamp in db.execute(query, [shard['source_id'], *source_lines]):
                targets[int(line)] = (host, proc, timestamp)
        items = []
        for row in selected:
            target = targets.get(int(lines[row]))
            if target is None or not target[0] or not target[1] or target[2] is None:
                missing.append([shard['source_id'], int(lines[row])])
                continue
            items.append((int(row), target))
        graph_batch, row_batch = [], []
        def score_ready(force=False):
            while len(graph_batch) >= GRAPH_BATCH_SIZE or (force and graph_batch):
                take = min(GRAPH_BATCH_SIZE, len(graph_batch))
                scores[row_batch[:take]] = batch_scorer.score(graph_batch[:take])
                del graph_batch[:take]
                del row_batch[:take]
        if graph_pool is None:
            for row, target in items:
                graph_batch.append(ego.build(*target))
                row_batch.append(row)
                score_ready()
        else:
            chunks = (items[start:start + GRAPH_BUILD_BATCH_SIZE]
                      for start in range(0, len(items), GRAPH_BUILD_BATCH_SIZE))
            pending = deque()
            chunk_count = (len(items) + GRAPH_BUILD_BATCH_SIZE - 1) // GRAPH_BUILD_BATCH_SIZE
            for _ in range(min(2 * GRAPH_CPU_WORKERS, chunk_count)):
                try:
                    pending.append(graph_pool.submit(_build_graph_chunk, next(chunks)))
                except StopIteration:
                    break
            while pending:
                future = pending.popleft()
                chunk_rows, chunk_graphs = future.result()
                row_batch.extend(chunk_rows)
                graph_batch.extend(chunk_graphs)
                score_ready()
                try:
                    pending.append(graph_pool.submit(_build_graph_chunk, next(chunks)))
                except StopIteration:
                    pass
        score_ready(force=True)
        if missing:
            atomic_json(output / f'{split}-missing-{index:04d}.json', missing)
            raise ValueError('Missing required graph targets')
        return scores

    def shard_cache_paths(output, index):
        directory = output / 'test_shards'
        directory.mkdir(parents=True, exist_ok=True)
        payload = directory / f'test-shard-{index:04d}.npz'
        return payload, payload.with_suffix('.receipt.json')

    def save_test_shard(output, index, shard, policy_hash, arrays):
        payload, receipt_path = shard_cache_paths(output, index)
        atomic_npz(payload, **arrays)
        atomic_json(receipt_path, dict(version=2, identity=checkpoint_identity(identity), policy_sha256=policy_hash,
            source_id=shard['source_id'], shard_sha256=shard['sha256'], rows=int(shard['rows']),
            payload_sha256=sha(payload)))

    def load_test_shard(output, index, shard, policy_hash):
        payload, receipt_path = shard_cache_paths(output, index)
        if not payload.exists() or not receipt_path.exists():
            return None
        receipt = json.loads(receipt_path.read_text())
        expected = dict(version=2, identity=checkpoint_identity(identity), policy_sha256=policy_hash,
                        source_id=shard['source_id'], shard_sha256=shard['sha256'], rows=int(shard['rows']))
        if any(receipt.get(k) != v for k, v in expected.items()) or sha(payload) != receipt.get('payload_sha256'):
            raise ValueError(f'TEST shard checkpoint identity/hash mismatch: {index}')
        if sha(data.path_for(shard)) != shard['sha256']:
            raise ValueError(f'Host shard content differs from frozen manifest: {index}')
        with np.load(payload, allow_pickle=False) as file:
            arrays = {name: file[name].copy() for name in file.files}
        required = {'y','lines','ml','graph','collision','old','new'}
        if set(arrays) != required or any(len(value) != int(shard['rows']) for value in arrays.values()):
            raise ValueError(f'Invalid TEST shard checkpoint shape: {index}')
        old, new, _ = predict(arrays['ml'], arrays['graph'], arrays['collision'], policy)
        if (not np.array_equal(old.astype(np.uint8), arrays['old'])
                or not np.array_equal(new.astype(np.uint8), arrays['new'])):
            raise ValueError(f'TEST shard checkpoint predictions do not reproduce: {index}')
        return arrays
    def validation_cache_paths(index):
        directory = output / 'validation_shards'
        directory.mkdir(parents=True, exist_ok=True)
        payload = directory / f'validation-shard-{index:04d}.npz'
        return payload, payload.with_suffix('.receipt.json')

    def save_validation_shard(index, shard, arrays):
        payload, receipt_path = validation_cache_paths(index)
        atomic_npz(payload, **arrays)
        atomic_json(receipt_path, dict(version=1, identity=checkpoint_identity(identity),
            source_id=shard['source_id'], shard_sha256=shard['sha256'], rows=int(shard['rows']),
            collision_rows=int(arrays['collision'].sum()), payload_sha256=sha(payload)))

    def load_validation_shard(index, shard):
        payload, receipt_path = validation_cache_paths(index)
        if not payload.exists() and not receipt_path.exists():
            return None
        if not payload.is_file() or not receipt_path.is_file():
            payload.unlink(missing_ok=True); receipt_path.unlink(missing_ok=True)
            return None
        receipt = json.loads(receipt_path.read_text())
        expected = dict(version=1, identity=checkpoint_identity(identity),
                        source_id=shard['source_id'], shard_sha256=shard['sha256'],
                        rows=int(shard['rows']))
        if any(receipt.get(key) != value for key,value in expected.items()):
            raise ValueError(f'Validation shard checkpoint identity mismatch: {index}')
        if sha(payload) != receipt.get('payload_sha256'):
            raise ValueError(f'Validation shard checkpoint checksum mismatch: {index}')
        if sha(data.path_for(shard)) != shard['sha256']:
            raise ValueError(f'Host validation shard differs from frozen manifest: {index}')
        with np.load(payload, allow_pickle=False) as file:
            arrays = {name:file[name].copy() for name in file.files}
        required = {'y','lines','ml','graph','collision','old','key_offsets','key_bytes'}
        if set(arrays) != required or any(len(arrays[name]) != int(shard['rows'])
                for name in ('y','lines','ml','graph','collision','old')):
            raise ValueError(f'Invalid validation checkpoint arrays: {index}')
        collision_rows = np.flatnonzero(arrays['collision'])
        offsets = arrays['key_offsets']
        if (len(offsets) != len(collision_rows)+1 or int(offsets[0]) != 0
                or int(offsets[-1]) != len(arrays['key_bytes']) or np.any(offsets[1:] < offsets[:-1])
                or len(collision_rows) != receipt.get('collision_rows')):
            raise ValueError(f'Invalid collision-key payload in validation checkpoint: {index}')
        routed = arrays['collision'] | ((arrays['ml'] >= LOW) & (arrays['ml'] < HIGH))
        if not np.all(np.isfinite(arrays['graph'][routed])):
            raise ValueError(f'Missing required graph scores in validation checkpoint: {index}')
        old = np.where((arrays['ml'] >= LOW) & (arrays['ml'] < HIGH),
                       arrays['graph'] >= GRAPH, arrays['ml'] >= HIGH).astype(np.uint8)
        if not np.array_equal(old, arrays['old']):
            raise ValueError(f'Validation checkpoint predictions do not reproduce: {index}')
        return arrays

    try:
        val_rows = []
        val_total = np.zeros(4,dtype=np.int64)
        val_count = 0
        for index, shard in enumerate(data.meta['shards']):
            if shard['split'] != 'val':
                continue
            cached = load_validation_shard(index, shard)
            if cached is None:
                x,y,lines = get_shard(data,shard)
                ml = host_scores(x)
                collision = np.fromiter((key_for_row(x,i) in registry for i in range(len(y))),
                                        dtype=bool,count=len(y))
                middle = (ml >= LOW) & (ml < HIGH)
                graph = graph_scores('val',index,shard,lines,collision|middle)
                old = np.where(middle, graph >= GRAPH, ml >= HIGH).astype(np.uint8)
                collision_rows = np.flatnonzero(collision)
                key_offsets = [0]
                key_parts = []
                for row in collision_rows:
                    key_parts.append(key_for_row(x,int(row)))
                    key_offsets.append(key_offsets[-1] + len(key_parts[-1]))
                key_bytes = np.frombuffer(b''.join(key_parts), dtype=np.uint8).copy()
                cached = dict(y=y.astype(np.uint8), lines=lines.astype(np.int64),
                    ml=ml.astype(np.float64), graph=graph.astype(np.float64),
                    collision=collision.astype(bool), old=old,
                    key_offsets=np.asarray(key_offsets,dtype=np.int64), key_bytes=key_bytes)
                save_validation_shard(index,shard,cached)
                status='computed'
            else:
                status='resumed'
            y=cached['y']; lines=cached['lines']; ml=cached['ml']; graph=cached['graph']
            collision=cached['collision']; old=cached['old'].astype(bool)
            val_total += counts(y,old)
            val_count += len(y)
            collision_rows = np.flatnonzero(collision)
            offsets = cached['key_offsets']; key_bytes = cached['key_bytes']
            for ordinal,row in enumerate(collision_rows):
                key = key_bytes[int(offsets[ordinal]):int(offsets[ordinal+1])].tobytes()
                val_rows.append(dict(key=key, label=int(y[row]), ml=float(ml[row]),
                    graph=float(graph[row]), source_id=shard['source_id'],
                    source_line=int(lines[row]), legacy_prediction=int(old[row])))
            print(f'validation shard {index}: {len(y)} rows, {int(collision.sum())} collisions [{status}]',flush=True)
        if val_count != data.meta['splits']['val']['rows']:
            raise ValueError('Validation row count mismatch')
        low = [r for r in val_rows if r['ml'] < LOW]
        high = [r for r in val_rows if r['ml'] >= HIGH]
        policy = select_policy(val_total, ([r['label'] for r in low],[r['graph'] for r in low]),
                              ([r['label'] for r in high],[r['graph'] for r in high]))
        for row in val_rows:
            if row['ml'] < LOW:
                row['candidate_prediction'] = int(row['graph'] >= policy['low_threshold'])
            elif row['ml'] >= HIGH:
                row['candidate_prediction'] = int(row['graph'] >= policy['high_threshold'])
            else:
                row['candidate_prediction'] = row['legacy_prediction']
        val_transitions = Counter()
        for row in val_rows:
            new = row['candidate_prediction'] if policy['accepted'] else row['legacy_prediction']
            if new != row['legacy_prediction']:
                val_transitions[('FP' if row['label']==0 else 'FN')+
                    ('_corrected' if new==row['label'] else '_introduced')] += 1
        frozen_path = output / 'frozen_policy.json'
        if frozen_path.exists():
            prior_frozen = json.loads(frozen_path.read_text())
            if prior_frozen.get('identity') != checkpoint_identity(identity):
                raise ValueError('Existing frozen policy belongs to a different model/data identity')
            if prior_frozen.get('policy') != policy:
                raise ValueError('Recomputed validation policy differs from the already frozen policy')
            policy = prior_frozen['policy']
        else:
            atomic_json(frozen_path, dict(identity=checkpoint_identity(identity), policy=policy))
        atomic_json(output / 'validation_metrics.json',dict(identity=identity,scorer=scorer,
            policy=policy,legacy=metrics(val_total),selected=(policy['validation_candidate'] if policy['accepted']
            else metrics(val_total)),transition_counts=dict(val_transitions),
            collision_events=len(val_rows),low_collision_events=len(low),high_collision_events=len(high)))
        per_vector_audit(registry,val_rows,policy,dict(bundle=bundle,path=output))
        # TEST is opened only after the validation policy has been frozen on disk.
        frozen = json.loads((output / 'frozen_policy.json').read_text())
        if frozen['identity'] != checkpoint_identity(identity) or frozen['policy'] != policy:
            raise ValueError('Frozen validation policy changed')
        test_old=np.zeros(4,dtype=np.int64);test_new=np.zeros(4,dtype=np.int64)
        transitions=Counter(); routing=Counter(); test_count=0
        policy_hash = policy_fingerprint(policy)
        errors_tmp=output/'test_errors_full_scores.jsonl.tmp'
        changes_tmp=output/'event_transition_audit.jsonl.tmp'
        with errors_tmp.open('w') as errors, changes_tmp.open('w') as changes:
            for index,shard in enumerate(data.meta['shards']):
                if shard['split'] != 'test':
                    continue
                cached = load_test_shard(output, index, shard, policy_hash)
                if cached is None:
                    x,y,lines=get_shard(data,shard)
                    ml=host_scores(x)
                    collision=np.fromiter((key_for_row(x,i) in registry for i in range(len(y))),
                                          dtype=bool,count=len(y))
                    middle=(ml>=LOW)&(ml<HIGH)
                    routed=middle | (collision if policy['accepted'] else np.zeros(len(y),dtype=bool))
                    graph=graph_scores('test',index,shard,lines,routed)
                    old,new,reason=predict(ml,graph,collision,policy)
                    if np.any((old!=new)&~collision) or np.any((old!=new)&middle):
                        raise ValueError('Middle/noncollision prediction changed')
                    cached=dict(y=y.astype(np.uint8), lines=lines.astype(np.int64),
                                ml=ml.astype(np.float64), graph=graph.astype(np.float64),
                                collision=collision.astype(bool), old=old.astype(np.uint8), new=new.astype(np.uint8))
                    save_test_shard(output,index,shard,policy_hash,cached)
                    status='computed'
                else:
                    y=cached['y']; lines=cached['lines']; ml=cached['ml']; graph=cached['graph']
                    collision=cached['collision']; old=cached['old'].astype(bool); new=cached['new'].astype(bool)
                    old,new,reason=predict(ml,graph,collision,policy)
                    status='resumed'
                test_old+=counts(y,old);test_new+=counts(y,new)
                test_count+=len(y)
                middle=(ml>=LOW)&(ml<HIGH)
                routed=middle | (collision if policy['accepted'] else np.zeros(len(y),dtype=bool))
                routing['legacy_graph']+=int(middle.sum())
                routing['selected_graph']+=int(routed.sum())
                routing['newly_routed']+=int((routed&~middle).sum())
                for row in np.flatnonzero(new!=y):
                    errors.write(json.dumps(dict(source_id=shard['source_id'],source_line=int(lines[row]),
                        label=int(y[row]),ml_score=float(ml[row]),graph_score=float(graph[row])
                        if np.isfinite(graph[row]) else None,collision=bool(collision[row]),
                        route_reason=str(reason[row]),prediction=int(new[row]),
                        legacy_prediction=int(old[row]),error='FP' if y[row]==0 else 'FN'))+'\n')
                for row in np.flatnonzero(old!=new):
                    kind=('FP' if y[row]==0 else 'FN')+('_corrected' if new[row]==y[row] else '_introduced')
                    transitions[kind]+=1
                    changes.write(json.dumps(dict(source_id=shard['source_id'],source_line=int(lines[row]),
                        label=int(y[row]),ml_score=float(ml[row]),graph_score=float(graph[row]),
                        collision=True,old_prediction=int(old[row]),new_prediction=int(new[row]),
                        reason=str(reason[row]),transition=kind))+'\n')
                errors.flush(); changes.flush()
                print(f'TEST shard {index}: {len(y)} rows, {int(np.sum(old!=new))} changes [{status}]',flush=True)
        if test_count!=data.meta['splits']['test']['rows']:
            raise ValueError('TEST row count mismatch')
        old_c=metrics(test_old)['confusion'];new_c=metrics(test_new)['confusion']
        reconciled=dict(tp=old_c['tp']+transitions['FN_corrected']-transitions['FN_introduced'],
            fp=old_c['fp']-transitions['FP_corrected']+transitions['FP_introduced'],
            fn=old_c['fn']-transitions['FN_corrected']+transitions['FN_introduced'],
            tn=old_c['tn']+transitions['FP_corrected']-transitions['FP_introduced'])
        if new_c!=reconciled:
            raise ValueError('TEST transitions do not reconcile')
        atomic_json(output/'test_metrics.json',dict(identity=identity,policy=policy,
            legacy=metrics(test_old),selected=metrics(test_new),routing=dict(routing),
            transition_counts=dict(transitions)))
        lines=['# Conditional collision-aware LightGBM + exact-time GraphSAGE', '',
            'Policy selection used validation labels only. Low extra-FP budget: 10; '
            'high extra-FN budget: 10; minimum validation F1 gain: 0.0001. '
            'Middle band and non-collision decisions remained legacy. Model weights, '
            'Host preprocessing and exact-time graph were unchanged.', '',
            f"Selected: **{policy['selected']}**. Best validation candidate: low graph "
            f"threshold {policy['low_threshold']:.9g}, high veto threshold "
            f"{policy['high_threshold']:.9g}; low extra FP {policy['low_extra_fp']}, "
            f"FN rescued {policy['low_rescued_fn']}; high FP vetoed "
            f"{policy['high_vetoed_fp']}, FN introduced {policy['high_extra_fn']}.", '',
            '| Split | Policy | TP | FP | FN | TN | Precision | Recall | F1 | FPR |',
            '| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |']
        for name,old,new in [('Validation',metrics(val_total),policy['validation_candidate'] if policy['accepted'] else metrics(val_total)),
                             ('TEST',metrics(test_old),metrics(test_new))]:
            for tag,value in [('legacy',old),('selected',new)]:
                c=value['confusion']
                lines.append(f"| {name} | {tag} | {c['tp']} | {c['fp']} | {c['fn']} | {c['tn']} | "
                    f"{value['precision']:.6f} | {value['recall']:.6f} | {value['f1']:.6f} | {value['fpr']:.6f} |")
        lines+=['',f"Validation transitions: {dict(val_transitions)}; TEST transitions: {dict(transitions)}.",
            f"TEST GraphSAGE calls: {routing['legacy_graph']}→{routing['selected_graph']} "
            f"(+{routing['newly_routed']}).",'',
            'The validation-approved policy reduces FP and improves F1, but requires '
            'substantially more GraphSAGE inference. Treat it as an experimental '
            'candidate rather than an unqualified deployment recommendation.', '',
            'Per-vector TRAIN counts, validation score distributions and errors are in '
            '`validation_per_vector_audit.jsonl`. Verified opposite-label identical '
            'graph encodings are marked as representation limitations; the check samples '
            'one pair per vector and identical score, so it is not an exhaustive census. '
            'All final TEST FP/FN and prediction transitions are exported separately. '
            'The legacy comparison uses this same retrained GraphSAGE checkpoint; historical '
            'v4 metrics used a different checkpoint and are not asserted as a reproducible baseline here.']
        (output/'summary.md').write_text('\n'.join(lines)+'\n')
        return dict(validation=metrics(val_total),test=metrics(test_new),policy=policy)
    finally:
        if graph_pool is not None:
            graph_pool.shutdown(wait=True, cancel_futures=True)
        if db is not None:
            db.close()


def main():
    parser=argparse.ArgumentParser()
    for name in ('manifest','bundle','output'):
        parser.add_argument('--'+name,required=True)
    for name in ('score-map','cache-source','registry-source','volume-root-ssd'):
        parser.add_argument('--'+name)
    args=parser.parse_args()
    run(args.manifest,args.bundle,args.output,score_map=args.score_map,
        cache_source=args.cache_source,registry_source=args.registry_source,
        volume_root_ssd=args.volume_root_ssd)


if __name__=='__main__':
    main()


In [ ]:
sys.path.insert(0, str(WORK))
from conditional_collision_runtime import run
if RESUME_FROM is not None:
    resume_source = Path(RESUME_FROM).resolve()
    if not resume_source.is_dir():
        raise FileNotFoundError(f'RESUME_FROM directory not found: {resume_source}')
    OUTPUT.mkdir(parents=True, exist_ok=True)
    copied = []
    for name in ('config.json','source_index.json','collision_registry.jsonl',
                 'collision_registry_summary.json','frozen_policy.json'):
        source = resume_source / name
        if source.is_file():
            shutil.copy2(source, OUTPUT / name); copied.append(name)
    for name in ('validation_shards','test_shards'):
        source = resume_source / name
        if source.is_dir():
            shutil.copytree(source, OUTPUT / name, dirs_exist_ok=True); copied.append(name)
    if not copied:
        raise FileNotFoundError('RESUME_FROM contains no reusable registry, validation or TEST checkpoints')
    print('Resume artifacts copied:', copied, 'from:', resume_source, flush=True)
volume_root = manifest.parent / 'volumes/ssd'
if not volume_root.is_dir():
    raise FileNotFoundError(f'Missing Host shard volume: {volume_root}')
result = run(manifest, bundle, OUTPUT, volume_root_ssd=volume_root,
             cache_source=score_cache, registry_source=OUTPUT)
print('Frozen validation policy:', result['policy']['selected'])


In [ ]:
import json, zipfile
from collections import Counter
required = ('config.json', 'source_index.json', 'collision_registry_summary.json',
            'collision_registry.jsonl', 'frozen_policy.json', 'validation_metrics.json',
            'validation_per_vector_audit.jsonl', 'representation_limitations.json',
            'test_metrics.json', 'test_errors_full_scores.jsonl',
            'event_transition_audit.jsonl', 'summary.md')
for name in required:
    if not (OUTPUT / name).is_file():
        raise FileNotFoundError(OUTPUT / name)
validation = json.loads((OUTPUT / 'validation_metrics.json').read_text())
test = json.loads((OUTPUT / 'test_metrics.json').read_text())
frozen = json.loads((OUTPUT / 'frozen_policy.json').read_text())
if frozen['policy'] != validation['policy'] or frozen['policy'] != test['policy']:
    raise ValueError('Validation-frozen policy identity changed')
errors = Counter()
error_keys = set()
for line in (OUTPUT / 'test_errors_full_scores.jsonl').open():
    row = json.loads(line)
    key = (row['source_id'], row['source_line'])
    if key in error_keys or row['label'] == row['prediction']:
        raise ValueError('Invalid or duplicate TEST error')
    error_keys.add(key)
    errors[row['error']] += 1
conf = test['selected']['confusion']
if errors != Counter({'FP': conf['fp'], 'FN': conf['fn']}):
    raise ValueError('Incomplete TEST FP/FN export')
transitions = Counter()
transition_keys = set()
for line in (OUTPUT / 'event_transition_audit.jsonl').open():
    row = json.loads(line)
    key = (row['source_id'], row['source_line'])
    if key in transition_keys or row['old_prediction'] == row['new_prediction']:
        raise ValueError('Invalid or duplicate TEST transition')
    transition_keys.add(key)
    transitions[row['transition']] += 1
if dict(transitions) != test['transition_counts']:
    raise ValueError('Transition counts do not reconcile')
archive = WORK / 'lgbm_exact_time_conditional_collision_results.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in required:
        z.write(OUTPUT / name, arcname=name)
with zipfile.ZipFile(archive) as z:
    if z.testzip() is not None:
        raise ValueError('Corrupt result archive')
print('Download:', archive)
print('TEST errors:', dict(errors), 'prediction changes:', len(transition_keys))


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
for split in ('validation', 'test'):
    d=json.loads((OUTPUT / f'{split}_metrics.json').read_text())
    old,new=d['legacy'],d['selected']
    print(split.upper(), 'legacy:',old['confusion'],'F1=',round(old['f1'],6))
    print(split.upper(), 'selected:',new['confusion'],'F1=',round(new['f1'],6))
    fig,ax=plt.subplots(1,2,figsize=(10,3.5))
    ax[0].bar(['Legacy','Selected'],[old['f1'],new['f1']]);ax[0].set_title(split+' F1')
    ax[1].bar(['Legacy FP','Selected FP','Legacy FN','Selected FN'],
              [old['confusion']['fp'],new['confusion']['fp'],old['confusion']['fn'],new['confusion']['fn']])
    ax[1].tick_params(axis='x',rotation=30);ax[1].set_title(split+' errors')
    plt.tight_layout();plt.show()
display(Markdown((OUTPUT/'summary.md').read_text()))
